# DeepScan: Production-Grade Side-Scan Sonar AI Training
**Smart India Hackathon 2026 | Problem Statement 26057**

This notebook uses **Heavy Acoustic Augmentation** and Domain Adaptation to train a highly robust YOLOv8 model for AUV Edge deployment, even on small datasets.

In [ ]:
!pip install ultralytics roboflow opencv-python
import torch
print("GPU Available:", torch.cuda.is_available())

### 1. Download Dataset via Roboflow
Replace the API key with your own from Roboflow Universe.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY_HERE")
project = rf.workspace("underwater-object-detection").project("side-scan-sonar-debris")
version = project.version(1)
dataset = version.download("yolov8")

### 2. Train with Heavy Acoustic Augmentation
Standard YOLO crushes sonar data. We use massive augmentations to simulate murky water, varying sonar frequencies, and AUV roll/pitch.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 small model (optimal for Nvidia Jetson Edge deployment)
model = YOLO('yolov8s.pt')

# Train the model with acoustic physics parameters
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="models",
    name="sss_detector_v1",
    
    # --- HEAVY ACOUSTIC AUGMENTATION ---
    mosaic=1.0,         # Stitches images to teach scale variation
    mixup=0.2,          # Blends anomalies into different backgrounds
    hsv_h=0.015,        # Simulates changing sonar ping frequencies
    hsv_s=0.7, 
    hsv_v=0.4,          # Simulates attenuation (darkness) in deep water
    degrees=10.0,       # Simulates AUV rolling due to currents
    translate=0.1,      # Simulates AUV shifting
    flipud=0.5,         # Common in bidirectional sonar scans
    fliplr=0.5
)

### 3. Export for Deployment
Once training is complete, download `models/sss_detector_v1/weights/best.pt` and place it in your local React/FastAPI project!